# Video -> Gaussian Splat (free Colab test)
Runs `pipeline/run.sh` from [realestatewalkthrough](https://github.com/tschnelli/realestatewalkthrough) on the sample video.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Then run the cells top to bottom.
Cells 2 and 4 take a while (installs, training). If a cell fails, copy the error text back.

In [ ]:
# 1. Confirm a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install COLMAP, ffmpeg and nerfstudio (several minutes)
!apt-get -qq update && apt-get -qq install -y colmap ffmpeg > /dev/null
!pip install -q nerfstudio
!colmap -h | head -1 && ns-train --help | head -1

In [ ]:
# 3. Get the project (and sample video)
!rm -rf /content/rew && git clone -q https://github.com/tschnelli/realestatewalkthrough /content/rew
!ls -la /content/rew/samples

In [ ]:
# 4. Run the pipeline: frames -> COLMAP poses -> splatfacto training -> export
# Apt COLMAP has no CUDA, so feature matching runs on CPU (--no-gpu). Fine for a short low-res clip.
# Exhaustive matching compares every frame pair: slower, but far better registration on a walkthrough.
# 7000 iterations keeps the free T4 session short; raise ITERS for better quality.
import os
os.environ["QT_QPA_PLATFORM"] = "offscreen"      # COLMAP is headless here
os.environ["ITERS"] = "7000"
os.environ["PROCESS_ARGS"] = "--no-gpu --sfm-tool colmap --matching-method exhaustive"
# gsplat compiles CUDA code on first use; parallel compiles can exhaust Colab RAM ("Killed").
os.environ["MAX_JOBS"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"       # T4 only, avoids compiling for every GPU generation
!cd /content/rew && bash pipeline/run.sh samples/cabin-walkthrough.mov /content/out 200 2>&1 | tail -n 60
!free -h | head -2

In [ ]:
# 5. Check how many frames COLMAP registered (low numbers = poor capture)
import json, glob, os
f = glob.glob('/content/out/data/transforms.json')
t = json.load(open(f[0])) if f else {}
total = len(glob.glob('/content/out/data/images/*'))
print(f"registered frames: {len(t.get('frames', []))} of {total} extracted")

In [ ]:
# 6. Download the splat, then drop it in viewer/ as scene.ply and open viewer/?src=scene.ply
from google.colab import files
import glob
ply = sorted(glob.glob('/content/out/export/*.ply'))
if ply:
    print(ply)
    files.download(ply[0])
else:
    print('No .ply found: training did not finish. Re-run cell 4 and check the last lines of its output.')